# Hasil penelitian


## EDA

Bagian ini merangkum perbedaan jejak akuisisi citra dan dampak penanganannya pada probe sumber citra.


### Penanganan citra


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
from IPython.display import display

ROOT = Path(".")
sys.path.insert(0, str(ROOT / "src"))

from bdc.config import ACTIVE_MODEL, BACKBONES
from bdc.evaluation import ResearchMetrics
from bdc.visualization import ResultFigures

figures = ResultFigures()
acquisition = pd.read_csv(ROOT / "results" / "acquisition.csv")
display(acquisition.groupby("penanganan", as_index=False)["probe_auc"].mean())
figures.acquisition(acquisition)


## Feature extraction

Bagian ini mencantumkan delapan frozen backbone yang menghasilkan embedding untuk konfigurasi utama.


### Konfigurasi backbone


In [ ]:
models = pd.DataFrame([
    {"view": name, "backbone": item["model"], "input_size": item["size"]}
    for name, item in BACKBONES.items()
])
display(models)
print("Preset aktif:", ACTIVE_MODEL)


## Clustering

Setiap embedding diproyeksikan ke 32 dimensi, digabung, lalu direduksi ke 100 dimensi sebelum K-Means.


### Pemilihan jumlah cluster


In [ ]:
choice = pd.read_csv(ROOT / "results" / "k_selection.csv")
runs = pd.read_csv(ROOT / "results" / "k_stability.csv")
columns = [
    "k", "consensus_silhouette_median", "bootstrap_ci95_low",
    "bootstrap_ci95_high", "clusters_above_threshold", "chooseR_winner",
]
display(choice[columns])
clusters = pd.read_csv(ROOT / "results" / "cluster_stability.csv")
display(clusters[clusters["k"].isin([12, 14])][[
    "k", "cluster", "semantic_node", "jaccard_q2_5", "jaccard_median",
    "stability_status",
]])
figures.k_stability(choice, runs)


## External transfer

Bagian ini merangkum penerapan centroid internal pada citra Bangladesh tanpa fitting ulang di data tersebut.


### Transfer perangkat


In [ ]:
transfer = pd.read_csv(ROOT / "results" / "external_transfer_metrics.csv")
predictions = pd.read_csv(ROOT / "results" / "external_transfer_predictions.csv")
point = ResearchMetrics().strict_transfer(
    predictions["true_family"],
    predictions["strict_prediction_fused_k12"],
)
reported = transfer.loc[transfer["estimator"] == "fused_k12"].iloc[0]
display(pd.DataFrame([{
    "macro_top1_recomputed": point["macro_top1"],
    "macro_top1_reported": reported["strict_macro_top1"],
    "ci_low": reported["strict_macro_top1_ci_low"],
    "ci_high": reported["strict_macro_top1_ci_high"],
    "null_q95": reported["strict_macro_null_q95"],
}]))
figures.transfer(transfer)


## Analisis material

Bagian ini merangkum profil material visual pada cluster dan evaluasi DMS46 pada data internal serta external.


### Profil material per cluster


In [ ]:
profiles = pd.read_csv(ROOT / "results" / "cluster_material_profiles.csv")
figures.cluster_material(profiles)


### Evaluasi pada holdout internal


In [ ]:
material = pd.read_csv(ROOT / "results" / "material_holdout.csv")
columns = [
    "target", "n_holdout", "incremental_r2", "incremental_r2_ci_low",
    "incremental_r2_ci_high", "within_source_permutation_p",
]
display(material[columns])
per_material = pd.read_csv(ROOT / "results" / "material_per_material.csv")
display(per_material[["material", "mae", "mae_ci_low", "mae_ci_high", "spearman"]])
figures.material(material)


### Evaluasi material pada data external


In [ ]:
external = json.loads((ROOT / "results" / "external_material_metrics.json").read_text(encoding="utf-8"))
per_material = pd.read_csv(ROOT / "results" / "external_material_by_class.csv")
display(pd.DataFrame([external])[["primary_metric", "n_primary", "macro_auroc", "macro_average_precision"]])
display(per_material[["material", "support", "auroc", "average_precision"]])
figures.external_material(per_material)
